In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt, acos, pi
from pathlib import Path
OUTDIR = Path('corrected_delay_results')
OUTDIR.mkdir(exist_ok=True)

def delta_pi_xy(x, y, Dg1, Dr1, Dg2, Dr2):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    Dr_y = Dr1 + y * dDr
    Dg_y = Dg1 + y * dDg
    return -Dr_y + x * (Dr_y - Dg_y)

def equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2, tol=1e-12):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    a = dDr - dDg
    b = Dr1 - Dg1 - dDr
    c = -Dr1
    roots = []
    if abs(a) < tol:
        if abs(b) > tol:
            roots = [-c / b]
    else:
        roots_complex = np.roots([a, b, c])
        roots = [r.real for r in roots_complex if abs(r.imag) < 1e-09]
    roots = sorted([r for r in roots if r > tol and r < 1 - tol])
    return roots

def alpha_gamma_linear_delay(rho_star, Dg1, Dr1, Dg2, Dr2):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    Dr_star = Dr1 + rho_star * dDr
    Dg_star = Dg1 + rho_star * dDg
    pref = rho_star * (1.0 - rho_star)
    alpha = pref * (Dr_star - Dg_star)
    gamma = pref * (-dDr + rho_star * (dDr - dDg))
    return (alpha, gamma)

def corrected_tau_c(alpha, gamma):
    if gamma ** 2 <= alpha ** 2:
        return None
    omega = sqrt(gamma ** 2 - alpha ** 2)
    theta = acos(-alpha / gamma)
    if np.sign(np.sin(theta)) != np.sign(-omega / gamma):
        theta = 2 * pi - theta
    tau_c = theta / omega
    return (tau_c, omega, theta)

def classify_instantaneous_stability(alpha, gamma):
    val = alpha + gamma
    if val < 0:
        return 'stable'
    elif val > 0:
        return 'unstable'
    else:
        return 'neutral'

def print_delay_summary(Dg1, Dr1, Dg2, Dr2, label=''):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    print('=' * 72)
    print(label)
    print(f'G1 = G({Dg1}, {Dr1}), G2 = G({Dg2}, {Dr2})')
    print('Interior roots:', roots)
    for rho_star in roots:
        alpha, gamma = alpha_gamma_linear_delay(rho_star, Dg1, Dr1, Dg2, Dr2)
        hopf = corrected_tau_c(alpha, gamma)
        print()
        print(f'rho* = {rho_star:.10f}')
        print(f'alpha = {alpha:.10f}')
        print(f'gamma = {gamma:.10f}')
        print(f'alpha + gamma = {alpha + gamma:.10f}')
        print('tau=0 stability:', classify_instantaneous_stability(alpha, gamma))
        if hopf is None:
            print('Hopf condition gamma^2 > alpha^2: not satisfied')
        else:
            tau_c, omega_c, theta_c = hopf
            print('Hopf condition gamma^2 > alpha^2: satisfied')
            print(f'omega_c = {omega_c:.10f}')
            print(f'theta_c = {theta_c:.10f}')
            print(f'tau_c = {tau_c:.10f}')
    print('=' * 72)

def simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau, history_value, perturbation=0.001, tmax=500.0, dt=0.01, clip=True):
    n = int(tmax / dt) + 1
    t = np.linspace(0.0, tmax, n)
    rho = np.zeros(n)
    rho[0] = history_value + perturbation
    if clip:
        rho[0] = np.clip(rho[0], 0.0, 1.0)

    def get_delayed_value(time_now, i):
        td = time_now - tau
        if td <= 0:
            return history_value
        pos = td / dt
        j = int(np.floor(pos))
        frac = pos - j
        if j < 0:
            return history_value
        if j + 1 >= i:
            return rho[i]
        return (1.0 - frac) * rho[j] + frac * rho[j + 1]
    for i in range(n - 1):
        x = rho[i]
        y = get_delayed_value(t[i], i)
        dpi = delta_pi_xy(x, y, Dg1, Dr1, Dg2, Dr2)
        drift = x * (1.0 - x) * dpi
        rho[i + 1] = rho[i] + dt * drift
        if clip:
            rho[i + 1] = np.clip(rho[i + 1], 0.0, 1.0)
    return (t, rho)

def late_stats(t, rho, discard_fraction=0.6):
    start = int(discard_fraction * len(rho))
    late = rho[start:]
    return {'mean': float(np.mean(late)), 'std': float(np.std(late)), 'min': float(np.min(late)), 'max': float(np.max(late)), 'amplitude': float(0.5 * (np.max(late) - np.min(late))), 'final': float(rho[-1])}

def plot_dde_timeseries_across_threshold(Dg1, Dr1, Dg2, Dr2, tau_values, tmax=500, dt=0.01, perturbation=0.001, filename='dde_timeseries.png'):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    if not roots:
        raise ValueError('No interior equilibrium found.')
    chosen = None
    for r in roots:
        alpha, gamma = alpha_gamma_linear_delay(r, Dg1, Dr1, Dg2, Dr2)
        if alpha + gamma < 0:
            chosen = r
            break
    if chosen is None:
        raise ValueError('No stable interior equilibrium at tau=0 found.')
    rho_star = chosen
    plt.figure(figsize=(9, 5))
    for tau in tau_values:
        t, rho = simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau=tau, history_value=rho_star, perturbation=perturbation, tmax=tmax, dt=dt)
        plt.plot(t, rho, lw=1.5, label=f'$\\tau={tau:.2f}$')
    plt.axhline(rho_star, linestyle='--', linewidth=1.2, label=f'$\\rho^{{\\star}}={rho_star:.3f}$')
    plt.xlabel('time $t$')
    plt.ylabel('cooperation $\\rho(t)$')
    plt.title(f'Delayed replicator: $G_1=G({Dg1},{Dr1})$, $G_2=G({Dg2},{Dr2})$')
    plt.legend(frameon=False)
    plt.tight_layout()
    path = OUTDIR / filename
    plt.savefig(path, dpi=300)
    plt.show()
    return path

def plot_hopf_envelope(Dg1, Dr1, Dg2, Dr2, tau_min=0.0, tau_max=8.0, n_tau=80, tmax=1000, dt=0.02, discard_fraction=0.65, perturbation=0.001, filename='hopf_envelope.png'):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    stable_roots = []
    for r in roots:
        alpha, gamma = alpha_gamma_linear_delay(r, Dg1, Dr1, Dg2, Dr2)
        if alpha + gamma < 0:
            stable_roots.append((r, alpha, gamma))
    if not stable_roots:
        raise ValueError('No stable interior equilibrium found.')
    rho_star, alpha, gamma = stable_roots[0]
    hopf = corrected_tau_c(alpha, gamma)
    tau_c = None
    if hopf is not None:
        tau_c = hopf[0]
    tau_grid = np.linspace(tau_min, tau_max, n_tau)
    rho_min = []
    rho_max = []
    rho_mean = []
    for tau in tau_grid:
        t, rho = simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau=tau, history_value=rho_star, perturbation=perturbation, tmax=tmax, dt=dt)
        st = late_stats(t, rho, discard_fraction=discard_fraction)
        rho_min.append(st['min'])
        rho_max.append(st['max'])
        rho_mean.append(st['mean'])
    plt.figure(figsize=(7, 5))
    plt.plot(tau_grid, rho_min, lw=1.8, label='$\\rho_{\\min}$')
    plt.plot(tau_grid, rho_max, lw=1.8, label='$\\rho_{\\max}$')
    plt.plot(tau_grid, rho_mean, lw=1.0, linestyle='--', label='late-time mean')
    plt.axhline(rho_star, linestyle=':', linewidth=1.2, label=f'$\\rho^{{\\star}}={rho_star:.3f}$')
    if tau_c is not None:
        plt.axvline(tau_c, linestyle='--', linewidth=1.2, label=f'$\\tau_c={tau_c:.2f}$')
    plt.xlabel('delay $\\tau$')
    plt.ylabel('late-time cooperation')
    plt.title(f'Hopf envelope: $G_1=G({Dg1},{Dr1})$, $G_2=G({Dg2},{Dr2})$')
    plt.legend(frameon=False)
    plt.tight_layout()
    path = OUTDIR / filename
    plt.savefig(path, dpi=300)
    plt.show()
    return path

def plot_delay_embedding(Dg1, Dr1, Dg2, Dr2, tau, tmax=800, dt=0.01, discard_fraction=0.5, perturbation=0.001, filename='delay_embedding.png'):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    stable_roots = []
    for r in roots:
        alpha, gamma = alpha_gamma_linear_delay(r, Dg1, Dr1, Dg2, Dr2)
        if alpha + gamma < 0:
            stable_roots.append(r)
    if not stable_roots:
        raise ValueError('No stable interior equilibrium found.')
    rho_star = stable_roots[0]
    t, rho = simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau=tau, history_value=rho_star, perturbation=perturbation, tmax=tmax, dt=dt)
    delay_steps = int(round(tau / dt))
    if delay_steps < 1:
        raise ValueError('tau too small relative to dt for delay embedding.')
    start = max(delay_steps + 1, int(discard_fraction * len(rho)))
    rho_now = rho[start:]
    rho_delay = rho[start - delay_steps:len(rho) - delay_steps]
    n = min(len(rho_now), len(rho_delay))
    rho_now = rho_now[:n]
    rho_delay = rho_delay[:n]
    plt.figure(figsize=(5, 5))
    plt.plot(rho_delay, rho_now, lw=1.2)
    plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1.0)
    plt.scatter([rho_star], [rho_star], s=50, marker='*', label=f'$\\rho^{{\\star}}$')
    plt.xlabel('$\\rho(t-\\tau)$')
    plt.ylabel('$\\rho(t)$')
    plt.title(f'Delay embedding, $\\tau={tau:.2f}$')
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.legend(frameon=False)
    plt.tight_layout()
    path = OUTDIR / filename
    plt.savefig(path, dpi=300)
    plt.show()
    return path

def has_delay_induced_hopf_for_some_G1(Dg2, Dr2, grid_G1):
    for Dg1 in grid_G1:
        for Dr1 in grid_G1:
            roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
            for rho_star in roots:
                alpha, gamma = alpha_gamma_linear_delay(rho_star, Dg1, Dr1, Dg2, Dr2)
                if alpha + gamma < 0 and gamma ** 2 > alpha ** 2:
                    return True
    return False

def plot_existence_map_G2(n_grid_G2=81, n_grid_G1=81, filename='existence_map_G2.png'):
    vals_G2 = np.linspace(-1, 1, n_grid_G2)
    vals_G1 = np.linspace(-1, 1, n_grid_G1)
    existence = np.zeros((n_grid_G2, n_grid_G2), dtype=int)
    for i, Dr2 in enumerate(vals_G2):
        print(f'Row {i + 1}/{n_grid_G2}')
        for j, Dg2 in enumerate(vals_G2):
            existence[i, j] = int(has_delay_induced_hopf_for_some_G1(Dg2, Dr2, vals_G1))
    plt.figure(figsize=(6, 5))
    plt.imshow(existence, extent=[-1, 1, -1, 1], origin='lower', aspect='equal', interpolation='nearest')
    plt.xlabel('$D_g^{(2)}$')
    plt.ylabel('$D_r^{(2)}$')
    plt.title('Existence of delay-induced Hopf for some $G_1$')
    plt.colorbar(label='existence')
    plt.tight_layout()
    path = OUTDIR / filename
    plt.savefig(path, dpi=300)
    plt.show()
    return (path, existence)

def payoff_matrix_from_DgDr(Dg, Dr):
    return np.array([[1.0, -Dr], [1.0 + Dg, 0.0]], dtype=float)

def mixed_game_from_y(y, Dg1, Dr1, Dg2, Dr2):
    G1 = payoff_matrix_from_DgDr(Dg1, Dr1)
    G2 = payoff_matrix_from_DgDr(Dg2, Dr2)
    return (1.0 - y) * G1 + y * G2

def simulate_mc_delayed_fermi(Dg1, Dr1, Dg2, Dr2, N=2500, k=4, beta=0.5, rho0=0.44, delay_steps=0, n_sweeps=1000, seed=0):
    rng = np.random.default_rng(seed)
    strategies = np.zeros(N, dtype=np.int8)
    nC = int(round(rho0 * N))
    strategies[:nC] = 1
    rng.shuffle(strategies)
    total_updates = int(n_sweeps * N)
    record_every = N
    rho_history_updates = []
    rho_records = []
    t_records = []
    rho_current = strategies.mean()
    rho_history_updates.append(rho_current)
    rho_records.append(rho_current)
    t_records.append(0.0)
    for step in range(1, total_updates + 1):
        delayed_index = step - delay_steps
        if delayed_index <= 0:
            y = rho_history_updates[0]
        else:
            y = rho_history_updates[delayed_index - 1]
        G = mixed_game_from_y(y, Dg1, Dr1, Dg2, Dr2)
        i = rng.integers(N)
        j = rng.integers(N)
        while j == i:
            j = rng.integers(N)
        si = strategies[i]
        sj = strategies[j]
        opp_i = rng.integers(N, size=k)
        opp_j = rng.integers(N, size=k)
        payoff_i = 0.0
        payoff_j = 0.0
        for o in opp_i:
            payoff_i += G[1 - si, 1 - strategies[o]] if False else 0.0

        def idx(s):
            return 0 if s == 1 else 1
        row_i = idx(si)
        row_j = idx(sj)
        payoff_i = sum((G[row_i, idx(strategies[o])] for o in opp_i))
        payoff_j = sum((G[row_j, idx(strategies[o])] for o in opp_j))
        p = 1.0 / (1.0 + np.exp(-beta * (payoff_j - payoff_i)))
        if rng.random() < p:
            if strategies[i] != strategies[j]:
                if strategies[j] == 1:
                    rho_current += 1.0 / N
                else:
                    rho_current -= 1.0 / N
                strategies[i] = strategies[j]
        rho_history_updates.append(rho_current)
        if step % record_every == 0:
            rho_records.append(rho_current)
            t_records.append(step / N)
    return (np.array(t_records), np.array(rho_records))

def corrected_mc_delay_steps(tau_c_rep, N, beta, k, fraction=1.0):
    tau_mc = 2.0 / (beta * k) * tau_c_rep
    r = int(round(N * fraction * tau_mc))
    return (r, fraction * tau_mc)

def plot_mc_delay_fractions(Dg1, Dr1, Dg2, Dr2, N=2500, k=4, beta=0.5, fractions=(0.1, 0.25, 0.5, 0.75, 1.0, 1.2), n_sweeps=800, seed=0, filename='mc_delay_fractions.png'):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    if not roots:
        raise ValueError('No interior equilibrium found.')
    stable = []
    for r in roots:
        alpha, gamma = alpha_gamma_linear_delay(r, Dg1, Dr1, Dg2, Dr2)
        hopf = corrected_tau_c(alpha, gamma)
        if alpha + gamma < 0 and hopf is not None:
            stable.append((r, alpha, gamma, hopf[0]))
    if not stable:
        raise ValueError('No stable Hopf-capable equilibrium found.')
    rho_star, alpha, gamma, tau_c_rep = stable[0]
    plt.figure(figsize=(9, 5))
    for idx, frac in enumerate(fractions):
        delay_steps, tau_mc = corrected_mc_delay_steps(tau_c_rep=tau_c_rep, N=N, beta=beta, k=k, fraction=frac)
        t, rho = simulate_mc_delayed_fermi(Dg1, Dr1, Dg2, Dr2, N=N, k=k, beta=beta, rho0=rho_star, delay_steps=delay_steps, n_sweeps=n_sweeps, seed=seed + idx)
        plt.plot(t, rho, lw=1.2, label=f'$\\tau/\\tau_c={frac}$, $r={delay_steps}$')
    plt.axhline(rho_star, linestyle='--', linewidth=1.1, label=f'$\\rho^{{\\star}}={rho_star:.3f}$')
    plt.xlabel('time $t$ in sweeps')
    plt.ylabel('cooperation $\\rho(t)$')
    plt.title(f'MC delayed Fermi, $\\beta={beta}$, $k={k}$, $N={N}$')
    plt.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    path = OUTDIR / filename
    plt.savefig(path, dpi=300)
    plt.show()
    print('MC bridge summary')
    print(f'rho* = {rho_star:.8f}')
    print(f'alpha = {alpha:.8f}')
    print(f'gamma = {gamma:.8f}')
    print(f'tau_c_rep = {tau_c_rep:.8f}')
    print(f'beta = {beta}, k = {k}, s = beta*k/2 = {beta * k / 2:.4f}')
    print(f'tau_c_MC in sweeps = {2 / (beta * k) * tau_c_rep:.8f}')
    print(f'r_c elementary updates = {N * (2 / (beta * k)) * tau_c_rep:.2f}')
    return path
if __name__ == '__main__':
    Dg1_a, Dr1_a = (-0.75, -0.4)
    Dg2, Dr2 = (1.0, 1.0)
    print_delay_summary(Dg1_a, Dr1_a, Dg2, Dr2, label='Example A: expected corrected tau_c around 4.20')
    roots_a = equilibrium_roots_linear_feedback(Dg1_a, Dr1_a, Dg2, Dr2)
    rho_star_a = roots_a[0]
    alpha_a, gamma_a = alpha_gamma_linear_delay(rho_star_a, Dg1_a, Dr1_a, Dg2, Dr2)
    tau_c_a, _, _ = corrected_tau_c(alpha_a, gamma_a)
    plot_dde_timeseries_across_threshold(Dg1_a, Dr1_a, Dg2, Dr2, tau_values=[0.0, 0.5 * tau_c_a, 0.95 * tau_c_a, 1.05 * tau_c_a, 1.25 * tau_c_a], tmax=500, dt=0.01, filename='dde_timeseries_example_A.png')
    plot_delay_embedding(Dg1_a, Dr1_a, Dg2, Dr2, tau=1.2 * tau_c_a, tmax=700, dt=0.01, filename='delay_embedding_example_A.png')
    Dg1_b, Dr1_b = (-0.9, -0.7)
    print_delay_summary(Dg1_b, Dr1_b, Dg2, Dr2, label='Example B: expected corrected tau_c around 3.43')
    plot_hopf_envelope(Dg1_b, Dr1_b, Dg2, Dr2, tau_min=0.0, tau_max=7.0, n_tau=70, tmax=900, dt=0.02, filename='hopf_envelope_example_B.png')
    plot_mc_delay_fractions(Dg1_b, Dr1_b, Dg2, Dr2, N=2500, k=4, beta=0.5, fractions=(0.1, 0.25, 0.5, 0.75, 1.0, 1.2), n_sweeps=350, seed=42, filename='mc_delay_fractions_example_B.png')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from math import sqrt, acos, pi
OUTDIR = Path('corrected_main_delay_figures')
OUTDIR.mkdir(exist_ok=True)
USE_FULL_LATEX = False
if USE_FULL_LATEX:
    plt.rcParams.update({'text.usetex': True, 'font.family': 'serif', 'font.serif': ['Computer Modern Roman'], 'axes.labelsize': 18, 'axes.titlesize': 18, 'xtick.labelsize': 15, 'ytick.labelsize': 15, 'legend.fontsize': 13, 'figure.titlesize': 18, 'text.latex.preamble': '\\usepackage{amsmath}'})
else:
    plt.rcParams.update({'text.usetex': False, 'mathtext.fontset': 'cm', 'font.family': 'serif', 'font.serif': ['CMU Serif', 'Computer Modern Roman', 'DejaVu Serif'], 'axes.labelsize': 18, 'axes.titlesize': 18, 'xtick.labelsize': 15, 'ytick.labelsize': 15, 'legend.fontsize': 13, 'figure.titlesize': 18})

def delta_pi_xy(x, y, Dg1, Dr1, Dg2, Dr2):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    Dr_y = Dr1 + y * dDr
    Dg_y = Dg1 + y * dDg
    return -Dr_y + x * (Dr_y - Dg_y)

def interior_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2, tol=1e-12):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    a = dDr - dDg
    b = Dr1 - Dg1 - dDr
    c = -Dr1
    roots = []
    if abs(a) < tol:
        if abs(b) > tol:
            roots = [-c / b]
    else:
        candidates = np.roots([a, b, c])
        for r in candidates:
            if abs(r.imag) < 1e-09:
                roots.append(r.real)
    return sorted([r for r in roots if tol < r < 1 - tol])

def alpha_gamma(rho_star, Dg1, Dr1, Dg2, Dr2):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    Dr_star = Dr1 + rho_star * dDr
    Dg_star = Dg1 + rho_star * dDg
    pref = rho_star * (1.0 - rho_star)
    alpha = pref * (Dr_star - Dg_star)
    gamma = pref * (-dDr + rho_star * (dDr - dDg))
    return (alpha, gamma)

def corrected_tau_c(alpha, gamma):
    if gamma ** 2 <= alpha ** 2:
        return None
    omega = sqrt(gamma ** 2 - alpha ** 2)
    theta = acos(-alpha / gamma)
    if np.sign(np.sin(theta)) != np.sign(-omega / gamma):
        theta = 2 * pi - theta
    tau_c = theta / omega
    return (tau_c, omega, theta)

def stable_hopf_equilibrium(Dg1, Dr1, Dg2, Dr2):
    roots = interior_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    for rho_star in roots:
        alpha, gamma = alpha_gamma(rho_star, Dg1, Dr1, Dg2, Dr2)
        hopf = corrected_tau_c(alpha, gamma)
        if alpha + gamma < 0 and hopf is not None:
            tau_c, omega, theta = hopf
            return {'rho_star': rho_star, 'alpha': alpha, 'gamma': gamma, 'tau_c': tau_c, 'omega': omega, 'theta': theta}
    return None

def simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau, history_value, perturbation=0.001, tmax=800.0, dt=0.01):
    n = int(tmax / dt) + 1
    t = np.linspace(0.0, tmax, n)
    rho = np.zeros(n)
    rho[0] = np.clip(history_value + perturbation, 0.0, 1.0)

    def delayed_value(time_now, i):
        td = time_now - tau
        if td <= 0:
            return history_value
        pos = td / dt
        j = int(np.floor(pos))
        frac = pos - j
        if j < 0:
            return history_value
        if j + 1 >= i:
            return rho[i]
        return (1.0 - frac) * rho[j] + frac * rho[j + 1]
    for i in range(n - 1):
        x = rho[i]
        y = delayed_value(t[i], i)
        dpi = delta_pi_xy(x, y, Dg1, Dr1, Dg2, Dr2)
        drift = x * (1.0 - x) * dpi
        rho[i + 1] = rho[i] + dt * drift
        rho[i + 1] = np.clip(rho[i + 1], 0.0, 1.0)
    return (t, rho)

def compute_tau_heatmap_G2_PD(n_grid=601):
    Dg2, Dr2 = (1.0, 1.0)
    Dg_vals = np.linspace(-1.0, 1.0, n_grid)
    Dr_vals = np.linspace(-1.0, 1.0, n_grid)
    tau_map = np.full((n_grid, n_grid), np.nan)
    for i, Dr1 in enumerate(Dr_vals):
        if i % 50 == 0:
            print(f'Heatmap row {i + 1}/{n_grid}')
        for j, Dg1 in enumerate(Dg_vals):
            info = stable_hopf_equilibrium(Dg1, Dr1, Dg2, Dr2)
            if info is not None:
                tau_map[i, j] = info['tau_c']
    return (Dg_vals, Dr_vals, tau_map)

def make_fig6_combined_high_quality(n_grid=601, cmap_name='plasma', filename_base='fig6_corrected_combined_HQ'):
    Dg_vals, Dr_vals, tau_map = compute_tau_heatmap_G2_PD(n_grid=n_grid)
    log_tau = np.log10(tau_map)
    finite_vals = log_tau[np.isfinite(log_tau)]
    vmin = np.percentile(finite_vals, 2)
    vmax = np.percentile(finite_vals, 98)
    Dg1, Dr1 = (-0.75, -0.4)
    Dg2, Dr2 = (1.0, 1.0)
    info = stable_hopf_equilibrium(Dg1, Dr1, Dg2, Dr2)
    if info is None:
        raise RuntimeError('No stable Hopf-capable equilibrium found.')
    rho_star = info['rho_star']
    tau_c = info['tau_c']
    tau = 1.2 * tau_c
    print()
    print('Embedding parameters')
    print(f'G1 = G({Dg1}, {Dr1})')
    print(f'G2 = G({Dg2}, {Dr2})')
    print(f'rho* = {rho_star:.8f}')
    print(f'tau_c = {tau_c:.8f}')
    print(f'tau used = {tau:.8f}')
    dt = 0.01
    t, rho = simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau=tau, history_value=rho_star, perturbation=0.001, tmax=900, dt=dt)
    delay_steps = int(round(tau / dt))
    start = max(delay_steps + 1, int(0.58 * len(rho)))
    rho_now = rho[start:]
    rho_delay = rho[start - delay_steps:len(rho) - delay_steps]
    n = min(len(rho_now), len(rho_delay))
    rho_now = rho_now[:n]
    rho_delay = rho_delay[:n]
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.0), gridspec_kw={'width_ratios': [1.05, 1.0]}, constrained_layout=True)
    ax = axes[0]
    ax.set_facecolor('#d9d9d9')
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color='#d9d9d9')
    im = ax.imshow(log_tau, origin='lower', extent=[-1, 1, -1, 1], aspect='equal', interpolation='bilinear', cmap=cmap, vmin=vmin, vmax=vmax, rasterized=True)
    ax.axhline(0, color='black', lw=1.0, ls='--', alpha=0.75)
    ax.axvline(0, color='black', lw=1.0, ls='--', alpha=0.75)
    ax.set_xlabel('$D_g^{(1)}$', labelpad=5)
    ax.set_ylabel('$D_r^{(1)}$', labelpad=5)
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    ax.tick_params(axis='both', which='major', length=5, width=1.1)
    ax.text(0.0, 0.48, 'no stable solutions' + '\n' + 'in $(0,1)$ for the RE', ha='center', va='center', fontsize=13, color='black')
    ax.text(-0.95, 0.88, '$G_2=G(1,1)$', ha='left', va='center', fontsize=14, color='black')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.035)
    cbar.set_label('$\\log_{10}(\\tau_c)$', fontsize=16)
    cbar.ax.tick_params(labelsize=13, length=4, width=1.0)
    ax = axes[1]
    ax.plot(rho_delay, rho_now, lw=1.7, solid_capstyle='round')
    ax.plot([0, 1], [0, 1], ls='--', lw=1.2, color='black', alpha=0.55)
    ax.scatter([rho_star], [rho_star], marker='*', s=150, zorder=5, color='black')
    ax.text(rho_star + 0.035, rho_star + 0.045, '$\\rho^\\ast\\;(G_{\\rm eq})$', fontsize=15, ha='left', va='center')
    ax.set_xlabel('$\\rho(t-\\tau)$', labelpad=5)
    ax.set_ylabel('$\\rho(t)$', labelpad=5)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(axis='both', which='major', length=5, width=1.1)
    ax.text(0.04, 0.94, f'$\\tau={tau:.2f}>\\tau_c\\simeq {tau_c:.2f}$', transform=ax.transAxes, ha='left', va='top', fontsize=14)
    axes[0].text(-0.16, 1.04, '\\textbf{a}' if USE_FULL_LATEX else '$\\mathbf{a}$', transform=axes[0].transAxes, fontsize=18, va='bottom')
    axes[1].text(-0.16, 1.04, '\\textbf{b}' if USE_FULL_LATEX else '$\\mathbf{b}$', transform=axes[1].transAxes, fontsize=18, va='bottom')
    pdf_path = OUTDIR / f'{filename_base}.pdf'
    svg_path = OUTDIR / f'{filename_base}.svg'
    png_path = OUTDIR / f'{filename_base}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(svg_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=700, bbox_inches='tight')
    plt.show()
    print()
    print('Saved:')
    print(pdf_path)
    print(svg_path)
    print(png_path)
    return (pdf_path, svg_path, png_path)
make_fig6_combined_high_quality(n_grid=601, cmap_name='plasma')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from math import sqrt, acos, pi
OUTDIR = Path('corrected_main_delay_figures')
OUTDIR.mkdir(exist_ok=True)
USE_FULL_LATEX = False
if USE_FULL_LATEX:
    plt.rcParams.update({'text.usetex': True, 'font.family': 'serif', 'font.serif': ['Computer Modern Roman'], 'axes.labelsize': 22, 'axes.titlesize': 20, 'xtick.labelsize': 17, 'ytick.labelsize': 17, 'legend.fontsize': 15, 'figure.titlesize': 20, 'text.latex.preamble': '\\usepackage{amsmath}'})
else:
    plt.rcParams.update({'text.usetex': False, 'mathtext.fontset': 'cm', 'font.family': 'serif', 'font.serif': ['CMU Serif', 'Computer Modern Roman', 'DejaVu Serif'], 'axes.labelsize': 22, 'axes.titlesize': 20, 'xtick.labelsize': 17, 'ytick.labelsize': 17, 'legend.fontsize': 15, 'figure.titlesize': 20})

def delta_pi_xy(x, y, Dg1, Dr1, Dg2, Dr2):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    Dr_y = Dr1 + y * dDr
    Dg_y = Dg1 + y * dDg
    return -Dr_y + x * (Dr_y - Dg_y)

def interior_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2, tol=1e-12):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    a = dDr - dDg
    b = Dr1 - Dg1 - dDr
    c = -Dr1
    roots = []
    if abs(a) < tol:
        if abs(b) > tol:
            roots = [-c / b]
    else:
        candidates = np.roots([a, b, c])
        for r in candidates:
            if abs(r.imag) < 1e-09:
                roots.append(r.real)
    return sorted([r for r in roots if tol < r < 1 - tol])

def alpha_gamma(rho_star, Dg1, Dr1, Dg2, Dr2):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    Dr_star = Dr1 + rho_star * dDr
    Dg_star = Dg1 + rho_star * dDg
    pref = rho_star * (1.0 - rho_star)
    alpha = pref * (Dr_star - Dg_star)
    gamma = pref * (-dDr + rho_star * (dDr - dDg))
    return (alpha, gamma)

def corrected_tau_c(alpha, gamma):
    if gamma ** 2 <= alpha ** 2:
        return None
    omega = sqrt(gamma ** 2 - alpha ** 2)
    theta = acos(-alpha / gamma)
    if np.sign(np.sin(theta)) != np.sign(-omega / gamma):
        theta = 2 * pi - theta
    tau_c = theta / omega
    return (tau_c, omega, theta)

def stable_hopf_equilibrium(Dg1, Dr1, Dg2, Dr2):
    roots = interior_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    for rho_star in roots:
        alpha, gamma = alpha_gamma(rho_star, Dg1, Dr1, Dg2, Dr2)
        hopf = corrected_tau_c(alpha, gamma)
        if alpha + gamma < 0 and hopf is not None:
            tau_c, omega, theta = hopf
            return {'rho_star': rho_star, 'alpha': alpha, 'gamma': gamma, 'tau_c': tau_c, 'omega': omega, 'theta': theta}
    return None

def simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau, history_value, perturbation=0.001, tmax=500.0, dt=0.01):
    n = int(tmax / dt) + 1
    t = np.linspace(0.0, tmax, n)
    rho = np.zeros(n)
    rho[0] = np.clip(history_value + perturbation, 0.0, 1.0)

    def delayed_value(time_now, i):
        td = time_now - tau
        if td <= 0:
            return history_value
        pos = td / dt
        j = int(np.floor(pos))
        frac = pos - j
        if j < 0:
            return history_value
        if j + 1 >= i:
            return rho[i]
        return (1.0 - frac) * rho[j] + frac * rho[j + 1]
    for i in range(n - 1):
        x = rho[i]
        y = delayed_value(t[i], i)
        dpi = delta_pi_xy(x, y, Dg1, Dr1, Dg2, Dr2)
        drift = x * (1.0 - x) * dpi
        rho[i + 1] = rho[i] + dt * drift
        rho[i + 1] = np.clip(rho[i + 1], 0.0, 1.0)
    return (t, rho)

def make_fig5_timeseries_high_quality(filename_base='fig5_corrected_delay_timeseries_HQ'):
    Dg1, Dr1 = (-0.75, -0.4)
    Dg2, Dr2 = (1.0, 1.0)
    info = stable_hopf_equilibrium(Dg1, Dr1, Dg2, Dr2)
    if info is None:
        raise RuntimeError('No stable Hopf-capable equilibrium found.')
    rho_star = info['rho_star']
    tau_c = info['tau_c']
    alpha = info['alpha']
    gamma = info['gamma']
    print('FIG. 5 PARAMETERS')
    print(f'G1 = G({Dg1}, {Dr1})')
    print(f'G2 = G({Dg2}, {Dr2})')
    print(f'rho* = {rho_star:.10f}')
    print(f'alpha = {alpha:.10f}')
    print(f'gamma = {gamma:.10f}')
    print(f'alpha + gamma = {alpha + gamma:.10f}')
    print(f'corrected tau_c = {tau_c:.10f}')
    tau_values = [0.0, 0.5 * tau_c, 0.95 * tau_c, 1.05 * tau_c, 1.25 * tau_c]
    labels = ['$\\tau=0$', f'$\\tau={0.5 * tau_c:.2f}$', f'$\\tau={0.95 * tau_c:.2f}$', f'$\\tau={1.05 * tau_c:.2f}$', f'$\\tau={1.25 * tau_c:.2f}$']
    fig, ax = plt.subplots(figsize=(8.2, 4.4), constrained_layout=True)
    for tau, lab in zip(tau_values, labels):
        t, rho = simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau=tau, history_value=rho_star, perturbation=0.001, tmax=500, dt=0.01)
        ax.plot(t, rho, lw=2.0, label=lab, solid_capstyle='round')
    ax.axhline(rho_star, linestyle='--', linewidth=1.5, color='black', alpha=0.75, label=f'$\\rho^\\ast={rho_star:.3f}$')
    ax.set_xlabel('$t$', labelpad=6)
    ax.set_ylabel('$\\rho(t)$', labelpad=8)
    ax.set_xlim(0, 500)
    ax.set_ylim(0, 1)
    ax.set_xticks([0, 100, 200, 300, 400, 500])
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.tick_params(axis='both', which='major', length=6, width=1.2)
    ax.text(0.98, 0.95, f'$G_1=G(-0.75,-0.40)$' + '\n' + f'$G_2=G(1,1)$' + '\n' + f'$\\tau_c\\simeq {tau_c:.2f}$', transform=ax.transAxes, ha='right', va='top', fontsize=16, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none', alpha=0.75))
    ax.legend(frameon=False, loc='upper left', ncol=1, handlelength=2.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    pdf_path = OUTDIR / f'{filename_base}.pdf'
    svg_path = OUTDIR / f'{filename_base}.svg'
    png_path = OUTDIR / f'{filename_base}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(svg_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=700, bbox_inches='tight')
    plt.show()
    print()
    print('Saved:')
    print(pdf_path)
    print(svg_path)
    print(png_path)
    return (pdf_path, svg_path, png_path)
make_fig5_timeseries_high_quality()
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt, acos, pi
from pathlib import Path
OUTDIR = Path('corrected_delay_results_HQ')
OUTDIR.mkdir(exist_ok=True)
USE_FULL_LATEX = False
if USE_FULL_LATEX:
    plt.rcParams.update({'text.usetex': True, 'font.family': 'serif', 'font.serif': ['Computer Modern Roman'], 'text.latex.preamble': '\\usepackage{amsmath}', 'axes.labelsize': 22, 'axes.titlesize': 20, 'xtick.labelsize': 17, 'ytick.labelsize': 17, 'legend.fontsize': 14, 'figure.titlesize': 20, 'axes.linewidth': 1.2})
else:
    plt.rcParams.update({'text.usetex': False, 'mathtext.fontset': 'cm', 'font.family': 'serif', 'font.serif': ['CMU Serif', 'Computer Modern Roman', 'DejaVu Serif'], 'axes.labelsize': 22, 'axes.titlesize': 20, 'xtick.labelsize': 17, 'ytick.labelsize': 17, 'legend.fontsize': 14, 'figure.titlesize': 20, 'axes.linewidth': 1.2})

def save_hq(fig, filename_base, outdir=OUTDIR):
    pdf_path = outdir / f'{filename_base}.pdf'
    svg_path = outdir / f'{filename_base}.svg'
    png_path = outdir / f'{filename_base}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(svg_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=700, bbox_inches='tight')
    print('\nSaved:')
    print(pdf_path)
    print(svg_path)
    print(png_path)
    return (pdf_path, svg_path, png_path)

def clean_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', length=6, width=1.2)

def delta_pi_xy(x, y, Dg1, Dr1, Dg2, Dr2):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    Dr_y = Dr1 + y * dDr
    Dg_y = Dg1 + y * dDg
    return -Dr_y + x * (Dr_y - Dg_y)

def equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2, tol=1e-12):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    a = dDr - dDg
    b = Dr1 - Dg1 - dDr
    c = -Dr1
    roots = []
    if abs(a) < tol:
        if abs(b) > tol:
            roots = [-c / b]
    else:
        roots_complex = np.roots([a, b, c])
        roots = [r.real for r in roots_complex if abs(r.imag) < 1e-09]
    roots = sorted([r for r in roots if r > tol and r < 1 - tol])
    return roots

def alpha_gamma_linear_delay(rho_star, Dg1, Dr1, Dg2, Dr2):
    dDr = Dr2 - Dr1
    dDg = Dg2 - Dg1
    Dr_star = Dr1 + rho_star * dDr
    Dg_star = Dg1 + rho_star * dDg
    pref = rho_star * (1.0 - rho_star)
    alpha = pref * (Dr_star - Dg_star)
    gamma = pref * (-dDr + rho_star * (dDr - dDg))
    return (alpha, gamma)

def corrected_tau_c(alpha, gamma):
    if gamma ** 2 <= alpha ** 2:
        return None
    omega = sqrt(gamma ** 2 - alpha ** 2)
    theta = acos(-alpha / gamma)
    if np.sign(np.sin(theta)) != np.sign(-omega / gamma):
        theta = 2 * pi - theta
    tau_c = theta / omega
    return (tau_c, omega, theta)

def classify_instantaneous_stability(alpha, gamma):
    val = alpha + gamma
    if val < 0:
        return 'stable'
    elif val > 0:
        return 'unstable'
    else:
        return 'neutral'

def print_delay_summary(Dg1, Dr1, Dg2, Dr2, label=''):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    print('=' * 72)
    print(label)
    print(f'G1 = G({Dg1}, {Dr1}), G2 = G({Dg2}, {Dr2})')
    print('Interior roots:', roots)
    for rho_star in roots:
        alpha, gamma = alpha_gamma_linear_delay(rho_star, Dg1, Dr1, Dg2, Dr2)
        hopf = corrected_tau_c(alpha, gamma)
        print()
        print(f'rho* = {rho_star:.10f}')
        print(f'alpha = {alpha:.10f}')
        print(f'gamma = {gamma:.10f}')
        print(f'alpha + gamma = {alpha + gamma:.10f}')
        print('tau=0 stability:', classify_instantaneous_stability(alpha, gamma))
        if hopf is None:
            print('Hopf condition gamma^2 > alpha^2: not satisfied')
        else:
            tau_c, omega_c, theta_c = hopf
            print('Hopf condition gamma^2 > alpha^2: satisfied')
            print(f'omega_c = {omega_c:.10f}')
            print(f'theta_c = {theta_c:.10f}')
            print(f'tau_c = {tau_c:.10f}')
    print('=' * 72)

def stable_hopf_equilibrium(Dg1, Dr1, Dg2, Dr2):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    for rho_star in roots:
        alpha, gamma = alpha_gamma_linear_delay(rho_star, Dg1, Dr1, Dg2, Dr2)
        hopf = corrected_tau_c(alpha, gamma)
        if alpha + gamma < 0 and hopf is not None:
            tau_c, omega_c, theta_c = hopf
            return {'rho_star': rho_star, 'alpha': alpha, 'gamma': gamma, 'tau_c': tau_c, 'omega_c': omega_c, 'theta_c': theta_c}
    return None

def simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau, history_value, perturbation=0.001, tmax=500.0, dt=0.01, clip=True):
    n = int(tmax / dt) + 1
    t = np.linspace(0.0, tmax, n)
    rho = np.zeros(n)
    rho[0] = history_value + perturbation
    if clip:
        rho[0] = np.clip(rho[0], 0.0, 1.0)

    def get_delayed_value(time_now, i):
        td = time_now - tau
        if td <= 0:
            return history_value
        pos = td / dt
        j = int(np.floor(pos))
        frac = pos - j
        if j < 0:
            return history_value
        if j + 1 >= i:
            return rho[i]
        return (1.0 - frac) * rho[j] + frac * rho[j + 1]
    for i in range(n - 1):
        x = rho[i]
        y = get_delayed_value(t[i], i)
        dpi = delta_pi_xy(x, y, Dg1, Dr1, Dg2, Dr2)
        drift = x * (1.0 - x) * dpi
        rho[i + 1] = rho[i] + dt * drift
        if clip:
            rho[i + 1] = np.clip(rho[i + 1], 0.0, 1.0)
    return (t, rho)

def late_stats(t, rho, discard_fraction=0.6):
    start = int(discard_fraction * len(rho))
    late = rho[start:]
    return {'mean': float(np.mean(late)), 'std': float(np.std(late)), 'min': float(np.min(late)), 'max': float(np.max(late)), 'amplitude': float(0.5 * (np.max(late) - np.min(late))), 'final': float(rho[-1])}

def plot_dde_timeseries_across_threshold_HQ(Dg1, Dr1, Dg2, Dr2, tau_values, tmax=500, dt=0.01, perturbation=0.001, filename_base='supp_dde_timeseries_HQ'):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    if not roots:
        raise ValueError('No interior equilibrium found.')
    chosen = None
    chosen_alpha = None
    chosen_gamma = None
    for r in roots:
        alpha, gamma = alpha_gamma_linear_delay(r, Dg1, Dr1, Dg2, Dr2)
        if alpha + gamma < 0:
            chosen = r
            chosen_alpha = alpha
            chosen_gamma = gamma
            break
    if chosen is None:
        raise ValueError('No stable interior equilibrium at tau=0 found.')
    rho_star = chosen
    hopf = corrected_tau_c(chosen_alpha, chosen_gamma)
    tau_c = hopf[0] if hopf is not None else None
    fig, ax = plt.subplots(figsize=(8.2, 4.4), constrained_layout=True)
    for tau in tau_values:
        t, rho = simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau=tau, history_value=rho_star, perturbation=perturbation, tmax=tmax, dt=dt)
        ax.plot(t, rho, lw=2.0, label=f'$\\tau={tau:.2f}$', solid_capstyle='round')
    ax.axhline(rho_star, linestyle='--', linewidth=1.5, color='black', alpha=0.75, label=f'$\\rho^\\ast={rho_star:.3f}$')
    ax.set_xlabel('$t$')
    ax.set_ylabel('$\\rho(t)$')
    ax.set_xlim(0, tmax)
    ax.set_ylim(0, 1)
    ax.set_xticks(np.linspace(0, tmax, 6))
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    clean_axes(ax)
    text = f'$G_1=G({Dg1:.2f},{Dr1:.2f})$' + '\n' + f'$G_2=G({Dg2:.0f},{Dr2:.0f})$'
    if tau_c is not None:
        text += '\n' + f'$\\tau_c\\simeq {tau_c:.2f}$'
    ax.text(0.98, 0.95, text, transform=ax.transAxes, ha='right', va='top', fontsize=16, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none', alpha=0.75))
    ax.legend(frameon=False, loc='upper left', handlelength=2.4)
    return save_hq(fig, filename_base)

def plot_hopf_envelope_HQ(Dg1, Dr1, Dg2, Dr2, tau_min=0.0, tau_max=8.0, n_tau=100, tmax=1000, dt=0.02, discard_fraction=0.65, perturbation=0.001, filename_base='supp_hopf_envelope_HQ'):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    stable_roots = []
    for r in roots:
        alpha, gamma = alpha_gamma_linear_delay(r, Dg1, Dr1, Dg2, Dr2)
        if alpha + gamma < 0:
            stable_roots.append((r, alpha, gamma))
    if not stable_roots:
        raise ValueError('No stable interior equilibrium found.')
    rho_star, alpha, gamma = stable_roots[0]
    hopf = corrected_tau_c(alpha, gamma)
    tau_c = None
    if hopf is not None:
        tau_c = hopf[0]
    tau_grid = np.linspace(tau_min, tau_max, n_tau)
    rho_min = []
    rho_max = []
    rho_mean = []
    for idx, tau in enumerate(tau_grid):
        if idx % 10 == 0:
            print(f'Envelope simulation {idx + 1}/{len(tau_grid)}')
        t, rho = simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau=tau, history_value=rho_star, perturbation=perturbation, tmax=tmax, dt=dt)
        st = late_stats(t, rho, discard_fraction=discard_fraction)
        rho_min.append(st['min'])
        rho_max.append(st['max'])
        rho_mean.append(st['mean'])
    fig, ax = plt.subplots(figsize=(7.2, 5.0), constrained_layout=True)
    ax.plot(tau_grid, rho_min, lw=2.4, label='$\\rho_{\\min}$')
    ax.plot(tau_grid, rho_max, lw=2.4, label='$\\rho_{\\max}$')
    ax.plot(tau_grid, rho_mean, lw=1.6, linestyle='--', color='black', alpha=0.7, label='late-time mean')
    ax.axhline(rho_star, linestyle=':', linewidth=1.8, color='black', alpha=0.8, label=f'$\\rho^\\ast={rho_star:.3f}$')
    if tau_c is not None:
        ax.axvline(tau_c, linestyle='--', linewidth=1.8, color='black', alpha=0.8, label=f'$\\tau_c={tau_c:.2f}$')
    ax.set_xlabel('$\\tau$')
    ax.set_ylabel('late-time cooperation')
    ax.set_xlim(tau_min, tau_max)
    ax.set_ylim(-0.03, 1.03)
    clean_axes(ax)
    ax.legend(frameon=False, loc='upper left', handlelength=2.4)
    ax.text(0.98, 0.95, f'$G_1=G({Dg1:.2f},{Dr1:.2f})$' + '\n' + f'$G_2=G({Dg2:.0f},{Dr2:.0f})$', transform=ax.transAxes, ha='right', va='top', fontsize=16, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none', alpha=0.75))
    return save_hq(fig, filename_base)

def plot_delay_embedding_HQ(Dg1, Dr1, Dg2, Dr2, tau, tmax=800, dt=0.01, discard_fraction=0.55, perturbation=0.001, filename_base='supp_delay_embedding_HQ'):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    stable_roots = []
    for r in roots:
        alpha, gamma = alpha_gamma_linear_delay(r, Dg1, Dr1, Dg2, Dr2)
        if alpha + gamma < 0:
            stable_roots.append((r, alpha, gamma))
    if not stable_roots:
        raise ValueError('No stable interior equilibrium found.')
    rho_star, alpha, gamma = stable_roots[0]
    hopf = corrected_tau_c(alpha, gamma)
    tau_c = hopf[0] if hopf is not None else None
    t, rho = simulate_delayed_replicator(Dg1, Dr1, Dg2, Dr2, tau=tau, history_value=rho_star, perturbation=perturbation, tmax=tmax, dt=dt)
    delay_steps = int(round(tau / dt))
    if delay_steps < 1:
        raise ValueError('tau too small relative to dt for delay embedding.')
    start = max(delay_steps + 1, int(discard_fraction * len(rho)))
    rho_now = rho[start:]
    rho_delay = rho[start - delay_steps:len(rho) - delay_steps]
    n = min(len(rho_now), len(rho_delay))
    rho_now = rho_now[:n]
    rho_delay = rho_delay[:n]
    fig, ax = plt.subplots(figsize=(5.4, 5.0), constrained_layout=True)
    ax.plot(rho_delay, rho_now, lw=2.0, solid_capstyle='round')
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1.3, color='black', alpha=0.55)
    ax.scatter([rho_star], [rho_star], s=150, marker='*', color='black', zorder=5)
    ax.text(rho_star + 0.035, rho_star + 0.045, '$\\rho^\\ast$', fontsize=17, ha='left', va='center')
    ax.set_xlabel('$\\rho(t-\\tau)$')
    ax.set_ylabel('$\\rho(t)$')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    clean_axes(ax)
    title_text = f'$\\tau={tau:.2f}$'
    if tau_c is not None:
        title_text += f'$>\\tau_c\\simeq {tau_c:.2f}$'
    ax.text(0.04, 0.96, title_text, transform=ax.transAxes, ha='left', va='top', fontsize=16)
    return save_hq(fig, filename_base)

def has_delay_induced_hopf_for_some_G1(Dg2, Dr2, grid_G1):
    for Dg1 in grid_G1:
        for Dr1 in grid_G1:
            roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
            for rho_star in roots:
                alpha, gamma = alpha_gamma_linear_delay(rho_star, Dg1, Dr1, Dg2, Dr2)
                if alpha + gamma < 0 and gamma ** 2 > alpha ** 2:
                    return True
    return False

def plot_existence_map_G2_HQ(n_grid_G2=101, n_grid_G1=101, filename_base='supp_existence_map_G2_HQ'):
    vals_G2 = np.linspace(-1, 1, n_grid_G2)
    vals_G1 = np.linspace(-1, 1, n_grid_G1)
    existence = np.zeros((n_grid_G2, n_grid_G2), dtype=int)
    for i, Dr2 in enumerate(vals_G2):
        print(f'Existence map row {i + 1}/{n_grid_G2}')
        for j, Dg2 in enumerate(vals_G2):
            existence[i, j] = int(has_delay_induced_hopf_for_some_G1(Dg2, Dr2, vals_G1))
    fig, ax = plt.subplots(figsize=(5.6, 5.0), constrained_layout=True)
    cmap = plt.get_cmap('viridis')
    im = ax.imshow(existence, extent=[-1, 1, -1, 1], origin='lower', aspect='equal', interpolation='nearest', cmap=cmap, rasterized=True)
    ax.axhline(0, color='black', lw=1.0, ls='--', alpha=0.75)
    ax.axvline(0, color='black', lw=1.0, ls='--', alpha=0.75)
    ax.set_xlabel('$D_g^{(2)}$')
    ax.set_ylabel('$D_r^{(2)}$')
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    clean_axes(ax)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.035)
    cbar.set_label('existence')
    cbar.ax.tick_params(labelsize=14)
    save_hq(fig, filename_base)
    return existence

def payoff_matrix_from_DgDr(Dg, Dr):
    return np.array([[1.0, -Dr], [1.0 + Dg, 0.0]], dtype=float)

def mixed_game_from_y(y, Dg1, Dr1, Dg2, Dr2):
    G1 = payoff_matrix_from_DgDr(Dg1, Dr1)
    G2 = payoff_matrix_from_DgDr(Dg2, Dr2)
    return (1.0 - y) * G1 + y * G2

def simulate_mc_delayed_fermi(Dg1, Dr1, Dg2, Dr2, N=2500, k=4, beta=0.5, rho0=0.44, delay_steps=0, n_sweeps=1000, seed=0):
    rng = np.random.default_rng(seed)
    strategies = np.zeros(N, dtype=np.int8)
    nC = int(round(rho0 * N))
    strategies[:nC] = 1
    rng.shuffle(strategies)
    total_updates = int(n_sweeps * N)
    record_every = N
    rho_history_updates = []
    rho_records = []
    t_records = []
    rho_current = strategies.mean()
    rho_history_updates.append(rho_current)
    rho_records.append(rho_current)
    t_records.append(0.0)

    def idx(s):
        return 0 if s == 1 else 1
    for step in range(1, total_updates + 1):
        delayed_index = step - delay_steps
        if delayed_index <= 0:
            y = rho_history_updates[0]
        else:
            y = rho_history_updates[delayed_index - 1]
        G = mixed_game_from_y(y, Dg1, Dr1, Dg2, Dr2)
        i = rng.integers(N)
        j = rng.integers(N)
        while j == i:
            j = rng.integers(N)
        si = strategies[i]
        sj = strategies[j]
        opp_i = rng.integers(N, size=k)
        opp_j = rng.integers(N, size=k)
        row_i = idx(si)
        row_j = idx(sj)
        payoff_i = sum((G[row_i, idx(strategies[o])] for o in opp_i))
        payoff_j = sum((G[row_j, idx(strategies[o])] for o in opp_j))
        p = 1.0 / (1.0 + np.exp(-beta * (payoff_j - payoff_i)))
        if rng.random() < p:
            if strategies[i] != strategies[j]:
                if strategies[j] == 1:
                    rho_current += 1.0 / N
                else:
                    rho_current -= 1.0 / N
                strategies[i] = strategies[j]
        rho_history_updates.append(rho_current)
        if step % record_every == 0:
            rho_records.append(rho_current)
            t_records.append(step / N)
    return (np.array(t_records), np.array(rho_records))

def corrected_mc_delay_steps(tau_c_rep, N, beta, k, fraction=1.0):
    tau_mc = 2.0 / (beta * k) * tau_c_rep
    r = int(round(N * fraction * tau_mc))
    return (r, fraction * tau_mc)

def plot_mc_delay_fractions_HQ(Dg1, Dr1, Dg2, Dr2, N=2500, k=4, beta=0.5, fractions=(0.1, 0.25, 0.5, 0.75, 1.0, 1.2), n_sweeps=350, seed=0, filename_base='supp_mc_delay_fractions_HQ'):
    roots = equilibrium_roots_linear_feedback(Dg1, Dr1, Dg2, Dr2)
    if not roots:
        raise ValueError('No interior equilibrium found.')
    stable = []
    for r in roots:
        alpha, gamma = alpha_gamma_linear_delay(r, Dg1, Dr1, Dg2, Dr2)
        hopf = corrected_tau_c(alpha, gamma)
        if alpha + gamma < 0 and hopf is not None:
            stable.append((r, alpha, gamma, hopf[0]))
    if not stable:
        raise ValueError('No stable Hopf-capable equilibrium found.')
    rho_star, alpha, gamma, tau_c_rep = stable[0]
    fig, ax = plt.subplots(figsize=(9.2, 4.8), constrained_layout=True)
    for idx, frac in enumerate(fractions):
        delay_steps, tau_mc = corrected_mc_delay_steps(tau_c_rep=tau_c_rep, N=N, beta=beta, k=k, fraction=frac)
        t, rho = simulate_mc_delayed_fermi(Dg1, Dr1, Dg2, Dr2, N=N, k=k, beta=beta, rho0=rho_star, delay_steps=delay_steps, n_sweeps=n_sweeps, seed=seed + idx)
        ax.plot(t, rho, lw=1.8, label=f'$\\tau/\\tau_c={frac}$, $r={delay_steps}$', solid_capstyle='round')
    ax.axhline(rho_star, linestyle='--', linewidth=1.5, color='black', alpha=0.75, label=f'$\\rho^\\ast={rho_star:.3f}$')
    ax.set_xlabel('$t$ in sweeps')
    ax.set_ylabel('$\\rho(t)$')
    ax.set_xlim(0, n_sweeps)
    ax.set_ylim(0.2, 0.75)
    clean_axes(ax)
    ax.legend(frameon=False, loc='upper left', fontsize=12, handlelength=2.2)
    ax.text(0.98, 0.95, f'$N={N}$, $k={k}$, $\\beta={beta}$' + '\n' + f'$G_1=G({Dg1:.2f},{Dr1:.2f})$' + '\n' + f'$G_2=G({Dg2:.0f},{Dr2:.0f})$', transform=ax.transAxes, ha='right', va='top', fontsize=15, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none', alpha=0.75))
    print('MC bridge summary')
    print(f'rho* = {rho_star:.8f}')
    print(f'alpha = {alpha:.8f}')
    print(f'gamma = {gamma:.8f}')
    print(f'tau_c_rep = {tau_c_rep:.8f}')
    print(f'beta = {beta}, k = {k}, s = beta*k/2 = {beta * k / 2:.4f}')
    print(f'tau_c_MC in sweeps = {2 / (beta * k) * tau_c_rep:.8f}')
    print(f'r_c elementary updates = {N * (2 / (beta * k)) * tau_c_rep:.2f}')
    return save_hq(fig, filename_base)
if __name__ == '__main__':
    Dg1_a, Dr1_a = (-0.75, -0.4)
    Dg2, Dr2 = (1.0, 1.0)
    print_delay_summary(Dg1_a, Dr1_a, Dg2, Dr2, label='Example A: corrected tau_c around 4.20')
    roots_a = equilibrium_roots_linear_feedback(Dg1_a, Dr1_a, Dg2, Dr2)
    rho_star_a = roots_a[0]
    alpha_a, gamma_a = alpha_gamma_linear_delay(rho_star_a, Dg1_a, Dr1_a, Dg2, Dr2)
    tau_c_a, _, _ = corrected_tau_c(alpha_a, gamma_a)
    plot_dde_timeseries_across_threshold_HQ(Dg1_a, Dr1_a, Dg2, Dr2, tau_values=[0.0, 0.5 * tau_c_a, 0.95 * tau_c_a, 1.05 * tau_c_a, 1.25 * tau_c_a], tmax=500, dt=0.01, filename_base='supp_dde_timeseries_example_A_HQ')
    plot_delay_embedding_HQ(Dg1_a, Dr1_a, Dg2, Dr2, tau=1.2 * tau_c_a, tmax=800, dt=0.01, filename_base='supp_delay_embedding_example_A_HQ')
    Dg1_b, Dr1_b = (-0.9, -0.7)
    print_delay_summary(Dg1_b, Dr1_b, Dg2, Dr2, label='Example B: corrected tau_c around 3.43')
    plot_hopf_envelope_HQ(Dg1_b, Dr1_b, Dg2, Dr2, tau_min=0.0, tau_max=7.0, n_tau=90, tmax=900, dt=0.02, filename_base='supp_hopf_envelope_example_B_HQ')
    plot_mc_delay_fractions_HQ(Dg1_b, Dr1_b, Dg2, Dr2, N=2500, k=4, beta=0.5, fractions=(0.1, 0.25, 0.5, 0.75, 1.0, 1.2), n_sweeps=350, seed=42, filename_base='supp_mc_delay_fractions_example_B_HQ')
